# IMTalker on RunPod (RTX 4090)

**Efficient Audio-driven Talking Face Generation with Implicit Motion Transfer**

- Paper: https://arxiv.org/abs/2511.22167
- Model: https://huggingface.co/cbsjtu01/IMTalker
- Repo:  https://github.com/cbsjtu01/IMTalker

This notebook walks through:
1. GPU + environment check
2. Installing `conda` (if missing) and creating the `IMTalker` env with **Python 3.10**
3. Installing PyTorch 2.3.1 (CUDA 12.1) + ffmpeg + project requirements
4. Cloning the IMTalker repo
5. Downloading pretrained checkpoints from Hugging Face
6. Running **audio-driven** inference
7. Running **video-driven** inference
8. Launching the Gradio demo (`app.py`)

> **Important — how cells run on RunPod**: A Jupyter notebook on RunPod uses the kernel that started Jupyter (usually the base `python`). `conda activate` does **not** persist across `!` shell cells, so every command that needs the `IMTalker` env is wrapped in `conda run -n IMTalker ...`. That way each shell cell uses the right interpreter without depending on shell state.

## 0. Sanity check — GPU, CUDA, disk

In [ ]:
!nvidia-smi

In [ ]:
!nvcc --version || echo "nvcc not in PATH — that's OK, we install the CUDA 12.1 PyTorch wheels."
!df -h /workspace 2>/dev/null || df -h /

## 1. Choose a working directory

On RunPod, `/workspace` is the persistent volume — anything outside it disappears when the pod stops. We do all work there.

In [ ]:
import os

WORKDIR = "/workspace" if os.path.isdir("/workspace") else os.path.expanduser("~")
os.chdir(WORKDIR)
print("Working directory:", os.getcwd())

## 2. Install Miniconda (skip if already present)

Most RunPod PyTorch templates ship without conda. We install Miniconda silently into `/workspace/miniconda3`. If `conda` is already on the PATH the cell is essentially a no-op.

In [ ]:
# Make `conda` available to subsequent ! cells by prepending its bin dir to PATH for this kernel.
import os, shutil, subprocess

candidates = [
    "/workspace/miniconda3/bin",
    os.path.expanduser("~/miniconda3/bin"),
    "/opt/conda/bin",
]
for c in candidates:
    if os.path.isfile(os.path.join(c, "conda")) and c not in os.environ["PATH"]:
        os.environ["PATH"] = c + ":" + os.environ["PATH"]

print("conda binary:", shutil.which("conda"))
subprocess.run(["conda", "--version"], check=True)

## 3. Create the `IMTalker` conda environment (Python 3.10)

Matches the README's `conda create -n IMTalker python=3.10`, with two RunPod-friendly tweaks:

- Auto-accepts Anaconda's Terms of Service for the default channels (newer conda versions block installs until you do).
- Uses `-c conda-forge --override-channels` so we don't even touch the Anaconda default channels — same Python, no ToS surface.

In [ ]:
%%bash
set -e
# Accept Anaconda ToS just in case (newer conda versions require this before using defaults).
# Safe to run multiple times; ignore failure if these commands aren't supported by the installed conda.
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main 2>/dev/null || true
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r    2>/dev/null || true

if conda env list | awk '{print $1}' | grep -qx "IMTalker"; then
    echo "Env 'IMTalker' already exists — skipping creation."
else
    # Use conda-forge exclusively to avoid the Anaconda default-channels ToS prompt entirely.
    conda create -y -n IMTalker -c conda-forge --override-channels python=3.10
fi
conda run -n IMTalker python --version


## 4. Install PyTorch 2.3.1 + cu121 and ffmpeg

These are the exact versions the README specifies.

In [ ]:
%%bash
set -e

# 1. Install pip into the IMTalker env (conda-forge's python doesn't include it by default).
conda install -y -n IMTalker -c conda-forge --override-channels pip

# 2. Verify pip is now available inside the env.
conda run -n IMTalker python -m pip --version

# 3. Upgrade pip using the env's own interpreter.
conda run -n IMTalker python -m pip install --upgrade pip

# 4. Install PyTorch 2.3.1 + cu121 (exact versions from the README).
conda run -n IMTalker python -m pip install \
    torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 \
    --index-url https://download.pytorch.org/whl/cu121

# 5. Sanity check.
conda run -n IMTalker python -c "import torch, sys; print('python:', sys.version.split()[0]); print('torch :', torch.__version__); print('cuda  :', torch.cuda.is_available())"

In [ ]:
#apply for 5090 GPU

In [ ]:
# %%bash
# set -e

# # Wipe the old CUDA 12.1 torch trio.
# conda run -n IMTalker python -m pip uninstall -y torch torchvision torchaudio

# # Install Blackwell-capable PyTorch (CUDA 12.8 wheels include sm_120 kernels).
# conda run -n IMTalker python -m pip install \
#     torch==2.7.0 torchvision==0.22.0 torchaudio==2.7.0 \
#     --index-url https://download.pytorch.org/whl/cu128

# # Verify the 5090 is now recognized.
# conda run -n IMTalker python -c "
# import torch
# print('torch:', torch.__version__)
# print('cuda  :', torch.version.cuda)
# print('device:', torch.cuda.get_device_name(0))
# print('arch  :', torch.cuda.get_device_capability(0))
# print('supported archs:', torch.cuda.get_arch_list())
# "

In [ ]:
%%bash
set -e
# ffmpeg via conda-forge (matches README)
conda install -y -n IMTalker -c conda-forge --override-channels ffmpeg
conda run -n IMTalker ffmpeg -version | head -n 1

Quick CUDA sanity check inside the env:

In [ ]:
!conda run -n IMTalker python -c "import torch; print('torch:', torch.__version__); print('cuda available:', torch.cuda.is_available()); print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')"

## 5. Clone the IMTalker repository

In [ ]:
%%bash
set -e
cd /workspace 2>/dev/null || cd ~
if [ ! -d IMTalker-inference-v1 ]; then
    git clone https://github.com/MoshiHead/IMTalker-inference-v1.git
else
    echo "IMTalker-inference-v1 repo already present — pulling latest."
    cd IMTalker-inference-v1 && git pull --ff-only || true
fi
ls IMTalker-inference-v1 | head

In [ ]:
import os

REPO_DIR = "/workspace/IMTalker-inference-v1" if os.path.isdir("/workspace/IMTalker-inference-v1") else os.path.expanduser("~/IMTalker-inference-v1")
os.chdir(REPO_DIR)
print("Now in:", os.getcwd())

## 6. Install project Python requirements

Uses the repo's `requirement.txt` (the filename in this project is singular — no `s`).

In [ ]:
%%bash
set -e
cd "${REPO_DIR:-/workspace/IMTalker-inference-v1}"
# Handle either spelling, just in case the repo renames the file later.
REQ_FILE="requirement.txt"
[ -f "$REQ_FILE" ] || REQ_FILE="requirements.txt"
echo "Using $REQ_FILE"
conda run -n IMTalker pip install -r "$REQ_FILE"

## 7. Download pretrained checkpoints

We pull them straight from the Hugging Face repo into `./checkpoints` to match the directory structure in the README:

```
./checkpoints
├── renderer.ckpt
├── generator.ckpt
└── wav2vec2-base-960h/
    ├── config.json
    ├── pytorch_model.bin
    └── ...
```

(The README notes `python app.py` will auto-download these, but it's faster and more reliable to pre-fetch them here.)

In [ ]:
!conda run -n IMTalker pip install -U huggingface_hub

In [ ]:
%%bash
set -e
cd "${REPO_DIR:-/workspace/IMTalker-inference-v1}"
mkdir -p checkpoints

# Download all repo files into ./checkpoints
conda run -n IMTalker hf download cbsjtu01/IMTalker \
    --repo-type model \
    --local-dir checkpoints

echo
echo "== checkpoints tree =="
ls -lh checkpoints
echo
ls -lh checkpoints/wav2vec2-base-960h 2>/dev/null || \
    echo "(wav2vec2 folder will appear once HF mirror sync is done)"

## 8. Audio-driven inference

Generates a talking face from a source image + an audio clip. Adjust the paths below to your own files; defaults point to the sample assets shipped with the repo.

In [ ]:
# === EDIT THESE TO POINT AT YOUR OWN INPUTS ============================
REF_IMAGE   = "./assets/source_1.png"   # portrait image
AUDIO_FILE  = "./assets/audio_1.wav"    # English audio works best (see README)
RES_DIR     = "./results/"
A_CFG_SCALE = 2
# =======================================================================

import os
os.makedirs(RES_DIR, exist_ok=True)
for p in (REF_IMAGE, AUDIO_FILE):
    print("OK  " if os.path.exists(p) else "MISSING", p)

In [ ]:
%cd /workspace/IMTalker-inference-v1/

In [ ]:
%cd /workspace/IMTalker-inference-v1/
!PYTHONPATH=/workspace/IMTalker-inference-v1 conda run -n IMTalker --no-capture-output python generator/generate.py \
    --ref_path "{REF_IMAGE}" \
    --aud_path "{AUDIO_FILE}" \
    --res_dir "{RES_DIR}" \
    --generator_path "./checkpoints/generator.ckpt" \
    --renderer_path "./checkpoints/renderer.ckpt" \
    --a_cfg_scale {A_CFG_SCALE} \
    --crop

In [ ]:
# %cd /workspace/IMTalker-inference-v1/
# !conda run -n IMTalker --no-capture-output python generator/generate.py \
#     --ref_path "{REF_IMAGE}" \
#     --aud_path "{AUDIO_FILE}" \
#     --res_dir "{RES_DIR}" \
#     --generator_path "./checkpoints/generator.ckpt" \
#     --renderer_path "./checkpoints/renderer.ckpt" \
#     --a_cfg_scale {A_CFG_SCALE} \
#     --crop

In [ ]:
# Preview the most recent generated video inline (if any).
import glob, os
from IPython.display import Video, display

vids = sorted(glob.glob(os.path.join(REPO_DIR, RES_DIR.lstrip('./'), "**/*.mp4"), recursive=True),
              key=os.path.getmtime, reverse=True)
if vids:
    print("Latest output:", vids[0])
    display(Video(vids[0], embed=True))
else:
    print("No .mp4 found yet in", RES_DIR)

## 9. Video-driven inference

Drives the source image with the motion of a reference video.

In [ ]:
# === EDIT THESE TO POINT AT YOUR OWN INPUTS ============================
SOURCE_IMAGE  = "./assets/source_image.jpg"
DRIVING_VIDEO = "./assets/driving_video.mp4"
SAVE_DIR      = "./results/"
# =======================================================================

import os
os.makedirs(SAVE_DIR, exist_ok=True)
for p in (SOURCE_IMAGE, DRIVING_VIDEO):
    print("OK  " if os.path.exists(p) else "MISSING", p)

In [ ]:
!PYTHONPATH=/workspace/IMTalker conda run -n IMTalker --no-capture-output python renderer/inference.py \
    --source_path "{SOURCE_IMAGE}" \
    --driving_path "{DRIVING_VIDEO}" \
    --save_path "{SAVE_DIR}" \
    --renderer_path "./checkpoints/renderer.ckpt" \
    --crop

## 10. (Optional) Launch the Gradio demo

Runs `app.py` from the repo. On RunPod, expose the port via the **HTTP services** panel (default Gradio port is `7860`) and click the generated public URL.

> This cell will keep running until you interrupt the kernel (■ in the toolbar). That's expected — Gradio is a long-lived server.

In [ ]:
# Bind to 0.0.0.0 so RunPod's proxy can reach it. The repo's app.py uses Gradio defaults;
# GRADIO_SERVER_NAME forces the bind address without editing the source.
import os
os.environ["GRADIO_SERVER_NAME"] = "0.0.0.0"
os.environ["GRADIO_SERVER_PORT"] = "7860"

!cd "$REPO_DIR" && GRADIO_SERVER_NAME=0.0.0.0 GRADIO_SERVER_PORT=7860 \
    conda run -n IMTalker --no-capture-output python app.py

## 11. Troubleshooting cheat-sheet

- **`CUDA out of memory`** — the model targets a 24 GB card; close other notebooks/processes. Run `!nvidia-smi` to see what's using VRAM.
- **`ModuleNotFoundError`** — you almost certainly ran a cell with the wrong kernel. Every shell cell here uses `conda run -n IMTalker ...`; don't drop the prefix.
- **HF download stalls** — re-run the `huggingface-cli download` cell; it resumes partial files. For gated/private repos, run `conda run -n IMTalker huggingface-cli login` first.
- **`ffmpeg: not found`** during inference — re-run the conda-forge install cell; the script writes mp4s through ffmpeg.
- **Gradio link is local-only** — make sure `GRADIO_SERVER_NAME=0.0.0.0` is set, then open RunPod's proxied URL for port `7860`.
- **Crops your image wrong** — drop the `--crop` flag if your source is already a tight 512×512 face crop.
- **Best results** — English audio, head-only portraits, plain or blurred background, per the README's *Best Practices* section.